# Verify nuScenes full-keyframes dataset for E1
Checks folder layout, metadata version, and Boston/Singapore scene counts
before committing to a full campaign run.

In [ ]:
import os, glob, json
from pathlib import Path

# 1) locate the dataset mount
roots = glob.glob('/kaggle/input/*')
print('mounted inputs:', roots)
for r in roots:
    print(f'\n--- top level of {r} ---')
    for p in sorted(glob.glob(r + '/*'))[:30]:
        print('  ', os.path.basename(p), '(dir)' if os.path.isdir(p) else '(file)')


In [ ]:
# 2) find the metadata directory (v1.0-trainval / v1.0-mini / etc) anywhere in the tree
meta_dirs = []
for dp, dns, fns in os.walk('/kaggle/input'):
    if 'scene.json' in fns and 'log.json' in fns:
        meta_dirs.append(dp)
print('metadata dirs (contain scene.json + log.json):')
for m in meta_dirs: print('  ', m)

# find where the images live
cam_dirs = glob.glob('/kaggle/input/**/samples/CAM_FRONT', recursive=True)
print('\nCAM_FRONT sample dirs:')
for c in cam_dirs:
    n = len(os.listdir(c)) if os.path.isdir(c) else 0
    print(f'   {c}  ({n} images)')


In [ ]:
# 3) parse scene/log and count Boston vs Singapore
if meta_dirs:
    M = meta_dirs[0]
    print('using metadata dir:', M)
    scenes = json.load(open(os.path.join(M, 'scene.json')))
    logs   = json.load(open(os.path.join(M, 'log.json')))
    logs_by = {l['token']: l for l in logs}
    from collections import Counter
    loc_counter = Counter()
    city = Counter()
    for s in scenes:
        log = logs_by.get(s['log_token'])
        if not log: continue
        loc = log.get('location','')
        loc_counter[loc]+=1
        if loc.startswith('boston'): city['boston']+=1
        elif loc.startswith('singapore'): city['singapore']+=1
    print('\ntotal scenes:', len(scenes))
    print('by exact location:', dict(loc_counter))
    print('\nBoston scenes   :', city['boston'])
    print('Singapore scenes:', city['singapore'])
    # rough per-scene keyframe count (samples per scene)
    samples = json.load(open(os.path.join(M,'sample.json')))
    print('total keyframe samples:', len(samples), '(~annotated frames across all scenes)')
else:
    print('NO metadata dir found - layout is non-standard, need to inspect manually')


In [ ]:
# 4) sanity: can we open one CAM_FRONT image path from sample_data?
if meta_dirs:
    M = meta_dirs[0]
    sd = json.load(open(os.path.join(M,'sample_data.json')))
    cam_front = [x for x in sd if x.get('filename','').startswith('samples/CAM_FRONT/')][:3]
    print('example CAM_FRONT filenames from sample_data.json:')
    for x in cam_front:
        print('  ', x['filename'])
        # try to resolve it under the dataset root
        hits = glob.glob('/kaggle/input/**/' + x['filename'], recursive=True)
        print('     resolves to:', hits[:1] if hits else 'NOT FOUND on disk')
